# CoxingCoachAI — Local Whisper Notebook

This Colab notebook prototypes the core pipeline for the coxswain training website app.

This edition does **not** require `OPENAI_API_KEY` for audio transcription. It uses Faster-Whisper locally in the Colab runtime.

Core behavior:

- Upload `.m4a`, `.wav`, `.mp3`, or other supported recordings.
- Transcribe with local Whisper and rowing vocabulary prompting.
- Review transcript metrics.
- Select one or more feedback focus areas, or leave the selection blank for general feedback.
- Run the ideal-world race simulator.
- Generate transparent rule-based post-race feedback.

## 1. Setup

Run this cell in Google Colab. A GPU is optional; the notebook automatically uses GPU when available and CPU otherwise.

In [ ]:
!python --version
!pip -q install -U faster-whisper pandas numpy

## 2. Imports and configuration

No API key is needed. The first model load downloads the selected Whisper model into the Colab runtime.

In [ ]:
from __future__ import annotations

import os
import re
import json
import math
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
import numpy as np

FOCUS_AREAS = {
    'communication_clarity': 'Communication clarity and economy of words',
    'tone': 'Tone and phase-appropriate intensity',
    'technical_calls': 'Technical calls and rowing vocabulary',
    'rhythm_timing': 'Rhythmic synchronization and call timing',
    'rate_management': 'Rate management and race rhythm',
    'tactical_awareness': 'Tactical awareness and race plan execution',
    'psychological_calibration': 'Psychological calibration and trust-building',
}

DEFAULT_SCENARIO = {
    'race_distance_m': 2000,
    'base_rate_spm': 32,
    'base_split_seconds': 105.0,
    'boat_class': '8+',
    'crew_level': 'Intermediate',
    'water_condition': 'Flat',
    'race_phase': 'Full 2k race',
}

SUPPORTED_AUDIO_EXTENSIONS = {'mp3', 'mp4', 'mpeg', 'mpga', 'm4a', 'wav', 'webm'}
MAX_LOCAL_AUDIO_MB = 100


## 3. Transcript metrics

These features provide transparent, non-LLM metrics that the feedback layer can use even if no API key is available.


In [ ]:
FILLER_WORDS = {'um', 'uh', 'like', 'you know', 'sort of', 'kind of', 'basically', 'actually', 'literally', 'just'}
TECHNICAL_TERMS = {'catch', 'finish', 'drive', 'recovery', 'legs', 'swing', 'body', 'arms', 'set', 'send', 'length', 'ratio', 'blade', 'square', 'feather', 'split', 'rate', 'rhythm', 'pressure'}
TACTICAL_TERMS = {'seat', 'bow ball', 'walk', 'move', 'through', 'open water', 'inside', 'outside', 'line', 'current', 'wind', 'wake', '500', '750', '1000', 'last'}
MOTIVATIONAL_TERMS = {'trust', 'commit', 'together', 'believe', 'now', 'fight', 'empty', 'go', 'send', 'hold', 'breathe'}

def words(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z']+", text.lower())

def count_terms(text: str, terms: set[str]) -> int:
    normalized = re.sub(r'\s+', ' ', text.strip().lower())
    return sum(len(re.findall(rf'\b{re.escape(term)}\b', normalized)) for term in terms)

def extract_transcript_metrics(text: str) -> dict[str, Any]:
    w = words(text)
    filler_count = count_terms(text, FILLER_WORDS)
    command_patterns = {
        'power_10': r'\b(power|move|push)\s*(ten|10)\b',
        'rate_shift': r'\b(rate|shift|up|bring it|take it)\s*(?:to|up)?\s*(\d{2})\b',
        'settle': r'\b(settle|base|lengthen|race rhythm)\b',
        'sprint': r'\b(sprint|last\s*500|empty|all out|finish it)\b',
    }
    return {
        'word_count': len(w),
        'estimated_call_count': max(1, len([c for c in re.split(r'[.!?;\n,]+', text) if c.strip()])),
        'filler_count': filler_count,
        'filler_rate_per_100_words': round(100 * filler_count / max(1, len(w)), 2),
        'technical_term_count': count_terms(text, TECHNICAL_TERMS),
        'tactical_term_count': count_terms(text, TACTICAL_TERMS),
        'motivational_term_count': count_terms(text, MOTIVATIONAL_TERMS),
        'detected_commands': {k: len(re.findall(v, text, flags=re.I)) for k, v in command_patterns.items()},
        'top_words': Counter([x for x in w if len(x) > 3]).most_common(8),
    }


## 4. Local transcription function

The function below uses Faster-Whisper. It preserves the same `result["text"]` interface used by the rest of the notebook.

In [ ]:
import torch
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
LOCAL_MODEL_NAME = "small.en" if DEVICE == "cuda" else "base.en"

print("Device:", DEVICE)
print("Compute type:", COMPUTE_TYPE)
print("Model:", LOCAL_MODEL_NAME)

local_whisper_model = WhisperModel(
    LOCAL_MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

ROWING_PROMPT = (
    "This is a rowing coxswain race or practice recording. Expected vocabulary includes "
    "coxswain, coxing, CoxBox, stroke rate, strokes per minute, split, power ten, "
    "power 10, settle, rhythm, catch, finish, drive, recovery, sprint, rate shift, "
    "bow pair, stern pair, port, starboard, shell, oars, bow ball, walk, and seats."
)


def transcribe_audio_file(audio_path: str | Path, model_name: str = LOCAL_MODEL_NAME) -> dict[str, Any]:
    path = Path(audio_path)
    if not path.exists():
        raise FileNotFoundError(path)

    ext = path.suffix.lower().lstrip(".")
    if ext not in SUPPORTED_AUDIO_EXTENSIONS:
        raise ValueError(f"Unsupported extension: {ext}")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_LOCAL_AUDIO_MB:
        raise ValueError(f"File is {size_mb:.1f} MB; expected <= {MAX_LOCAL_AUDIO_MB} MB")

    segments_generator, info = local_whisper_model.transcribe(
        str(path),
        language="en",
        beam_size=5,
        vad_filter=True,
        vad_parameters={"min_silence_duration_ms": 500},
        condition_on_previous_text=True,
        initial_prompt=ROWING_PROMPT,
    )
    segments = list(segments_generator)
    transcript = " ".join(segment.text.strip() for segment in segments if segment.text.strip()).strip()

    return {
        "text": transcript,
        "model": model_name,
        "device": DEVICE,
        "language": info.language,
        "language_probability": getattr(info, "language_probability", None),
        "segments": [
            {
                "start_seconds": round(float(segment.start), 2),
                "end_seconds": round(float(segment.end), 2),
                "text": segment.text.strip(),
            }
            for segment in segments
            if segment.text.strip()
        ],
        "source_file": path.name,
    }

## 5. Ideal-world race simulator

The simulator maps transcript-detected commands into telemetry. In the ideal-world MVP, power 10s and rate shifts always work as instructed.


In [ ]:
@dataclass(frozen=True)
class RaceEvent:
    meter: int
    event_type: str
    description: str
    rate_delta: float = 0.0
    split_delta: float = 0.0
    duration_m: int = 150

def split_calls(text: str) -> list[str]:
    calls = []
    for chunk in re.split(r'[.!?\n]+', text):
        calls.extend([c.strip() for c in chunk.split(',') if c.strip()])
    return calls or [text.strip()]

def extract_race_events(text: str, race_distance_m: int) -> list[RaceEvent]:
    calls = split_calls(text)
    events = []
    for i, call in enumerate(calls):
        meter = int((i / max(1, len(calls) - 1)) * race_distance_m)
        lower = call.lower()
        if re.search(r'\b(power|move|push)\s*(ten|10)\b', lower):
            events.append(RaceEvent(meter, 'power_10', 'Power 10 improves split for the next segment.', 1.0, -2.0, 250))
        rate_match = re.search(r'\b(?:rate|shift|up|take it|bring it)\s*(?:to|up)?\s*(\d{2})\b', lower)
        if rate_match:
            target_rate = float(rate_match.group(1))
            events.append(RaceEvent(meter, 'rate_shift', f'Rate shift toward {target_rate:.0f} spm.', target_rate, -1.0 if target_rate >= 34 else 0.5, 300))
        if re.search(r'\b(settle|base|lengthen|race rhythm)\b', lower):
            events.append(RaceEvent(meter, 'settle', 'Settle call returns crew toward base rhythm.', 0.0, 0.0, 300))
        if re.search(r'\b(sprint|last\s*500|empty|finish it|all out)\b', lower):
            events.append(RaceEvent(max(meter, int(race_distance_m * 0.75)), 'sprint', 'Sprint call increases rate and boat speed.', 3.0, -3.0, max(250, int(race_distance_m * 0.25))))
    return events

def event_effect(event: RaceEvent, meter: int) -> tuple[float, float]:
    if meter < event.meter or meter > event.meter + event.duration_m:
        return 0.0, 0.0
    progress = (meter - event.meter) / max(1, event.duration_m)
    strength = 0.35 + 0.65 * math.cos(progress * math.pi / 2)
    return event.rate_delta * strength, event.split_delta * strength

def simulate_race_from_transcript(transcript: str, scenario: dict[str, Any], step_m: int = 100) -> tuple[pd.DataFrame, pd.DataFrame]:
    distance = int(scenario.get('race_distance_m', 2000))
    base_rate = float(scenario.get('base_rate_spm', 32))
    base_split = float(scenario.get('base_split_seconds', 105.0))
    events = extract_race_events(transcript, distance)
    rows = []
    current_rate = base_rate
    for meter in range(0, distance + 1, step_m):
        rate = base_rate
        split = base_split
        active = []
        for event in events:
            if event.event_type == 'rate_shift' and event.meter <= meter <= event.meter + event.duration_m:
                rate = max(rate, float(event.rate_delta))
                split += event.split_delta
                active.append(event.event_type)
            elif event.event_type == 'settle' and event.meter <= meter <= event.meter + event.duration_m:
                rate = base_rate
                active.append(event.event_type)
            else:
                rd, sd = event_effect(event, meter)
                rate += rd
                split += sd
                if rd or sd:
                    active.append(event.event_type)
        current_rate = 0.65 * current_rate + 0.35 * rate
        rows.append({'meter': meter, 'stroke_rate_spm': round(current_rate, 1), 'split_seconds_per_500m': round(max(70, split), 1), 'active_events': ', '.join(sorted(set(active))) or 'base'})
    telemetry = pd.DataFrame(rows)
    events_df = pd.DataFrame([event.__dict__ for event in events])
    return telemetry, events_df


## 6. Rule-based feedback baseline

This baseline is useful for testing without an API key. The Streamlit app also includes an LLM feedback path.


In [ ]:
def resolve_focus(selected: list[str] | None) -> dict[str, str]:
    if not selected:
        return FOCUS_AREAS.copy()
    return {key: FOCUS_AREAS[key] for key in selected if key in FOCUS_AREAS}

def rule_based_feedback(transcript: str, selected_focus: list[str] | None, scenario: dict[str, Any]) -> str:
    metrics = extract_transcript_metrics(transcript)
    focus = resolve_focus(selected_focus)
    lines = ['# Coxing feedback', '']
    lines.append(f"Overall: about **{metrics['word_count']} words**, **{metrics['estimated_call_count']} call chunks**, and detected commands {metrics['detected_commands']}.")
    lines.append('')
    lines.append('## Focus-area feedback')
    for key, label in focus.items():
        if key == 'communication_clarity':
            lines.append(f"### {label}\nUse a cue-action-reason pattern. Estimated filler rate: {metrics['filler_rate_per_100_words']} per 100 words.")
        elif key == 'tone':
            lines.append(f"### {label}\nSeparate calm base-rhythm calls from urgent move calls and high-energy sprint calls.")
        elif key == 'technical_calls':
            lines.append(f"### {label}\nTie calls to mechanics: catch, legs, swing, finish, send, ratio.")
        elif key == 'rhythm_timing':
            lines.append(f"### {label}\nPractice placing short calls on consistent beats. Future audio analysis should compare call timing to catch/finish sounds.")
        elif key == 'rate_management':
            lines.append(f"### {label}\nState the target and transition window: 'In two, shift thirty-four.' Then confirm the hold.")
        elif key == 'tactical_awareness':
            lines.append(f"### {label}\nName the tactical reason for moves: opponent, bow ball, seat count, water, or race phase.")
        elif key == 'psychological_calibration':
            lines.append(f"### {label}\nBuild trust with controllable cues: 'Trust rhythm, legs together, now.'")
        lines.append('')
    lines.append('## Off-water drill')
    lines.append('Record a 3-minute race script against a metronome. Remove every phrase that does not provide a cue, action, or tactical reason.')
    return '\n'.join(lines)


## 7. Demo run

Use this transcript to test the simulator and feedback.


In [ ]:
demo_transcript = """
Sit ready. Attention. Row. Build one, build two, lengthen, swing together. Shift to thirty-four in two. That's one, that's two, now settle thirty-four. Eyes in the boat. Legs together. Send it. Power ten here to move through them. One, legs. Two, swing. Three, sharp catches. Four, hold the rate. Five, together. Six, breathe. Seven, send. Eight, walk. Nine, bow ball. Ten, now hold that split. We are taking seats. In the last 500, sprint in two. One. Two. Up two beats. Empty the tanks. Clean finishes, together, now.
""".strip()

metrics = extract_transcript_metrics(demo_transcript)
telemetry, events = simulate_race_from_transcript(demo_transcript, DEFAULT_SCENARIO)
print(json.dumps(metrics, indent=2))
display(events)
display(telemetry.head())
display(telemetry.tail())
print(rule_based_feedback(demo_transcript, ['communication_clarity', 'rate_management'], DEFAULT_SCENARIO))


## 8. Upload and transcribe an audio file in Colab

Run the cell below, choose an `.m4a` or another supported recording, and wait for local transcription. No API key is required.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No audio file was uploaded.")

audio_path = next(iter(uploaded.keys()))
result = transcribe_audio_file(audio_path)
transcript = result["text"]

print("Model:", result["model"])
print("Device:", result["device"])
print("Language:", result["language"])
if result["language_probability"] is not None:
    print("Language confidence:", f"{result['language_probability']:.2%}")

print("\nTRANSCRIPT\n")
print(transcript)

if result["segments"]:
    display(pd.DataFrame(result["segments"]))

print("\nCOXSWAIN FEEDBACK\n")
print(rule_based_feedback(transcript, None, DEFAULT_SCENARIO))

## 9. Next engineering milestones

1. Compare `tiny.en`, `base.en`, and `small.en` on a labeled coxing-audio test set.
2. Add word-level timestamps and confidence-based transcript review.
3. Add audio-event detection for oarlock, catch, and finish sounds.
4. Add a distance/meters bar and simple 2D shell animation.
5. Add scenario templates for starts, settles, mid-race moves, sprints, and head races.
6. Add user accounts and recording history only after privacy and consent are designed.